# Dynamic Tool Generation (ToolFactory) | Domain Applications

In [1]:
from langchain_openai import ChatOpenAI
from dataclasses import dataclass
from typing import Dict, List, Any, Callable, Optional
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
@dataclass
class ToolSpec:
    name: str
    description: str
    input_type: str
    output_type: str
    test_cases: List[Dict[str, Any]]

class ToolFactory:
    def __init__(self, max_attempts: int = 3):
        self.registry: Dict[str, Callable] = {}
        self.max_attempts = max_attempts

    def _generate_code(self, spec: ToolSpec, error_feedback: str = "") -> str:
        feedback = f"\nPrevious attempt failed: {error_feedback}. Fix the issue and regenerate." if error_feedback else ""
        resp = model.invoke(
            f"Write Python function `{spec.name}`: {spec.description}\n"
            f"Input: {spec.input_type} Output: {spec.output_type}\n"
            f"ONLY the function def, no imports outside stdlib.{feedback}")
        code = resp.content.strip()
        if "```" in code:
            code = code.split("```python")[-1].split("```")[0] if "```python" in code \
                else code.split("```")[1].split("```")[0]
        return code.strip()

    def _test_code(self, code: str, spec: ToolSpec) -> tuple[bool, Callable | None, list[str]]:
        """Execute generated code and run test cases with real assertions."""
        log, ns = [], {}
        try:
            exec(code, ns)
        except Exception as e:
            return False, None, [f"COMPILE ERROR: {e}"]
        func = ns.get(spec.name)
        if not func:
            return False, None, [f"Function '{spec.name}' not found"]
        all_passed = True
        for i, tc in enumerate(spec.test_cases):
            try:
                result = func(tc["input"])
                if result == tc["expected"]:
                    log.append(f"  Test {i+1}: PASS | {tc['input']!r} -> {result!r}")
                else:
                    log.append(f"  Test {i+1}: FAIL | expected {tc['expected']!r}, got {result!r}")
                    all_passed = False
            except Exception as e:
                log.append(f"  Test {i+1}: ERROR | {e}"); all_passed = False
        return all_passed, func, log

    def create_tool(self, spec: ToolSpec) -> Optional[Callable]:
        """Generate, test, retry on failure, register only if ALL tests pass."""
        errors = ""
        for attempt in range(1, self.max_attempts + 1):
            print(f"\n--- Attempt {attempt}/{self.max_attempts} for '{spec.name}' ---")
            code = self._generate_code(spec, errors)
            print(f"Code:\n{code}\n")
            passed, func, log = self._test_code(code, spec)
            print("Tests:\n" + "\n".join(log))
            if passed and func:
                self.registry[spec.name] = func
                print(f"REGISTERED '{spec.name}'")
                return func
            errors = f"Code:\n{code}\nFailures:\n" + "\n".join(log)
        print(f"FAILED after {self.max_attempts} attempts")
        return None

In [5]:
# --- Use case: CSV-row-to-JSON transformation tool ---
factory = ToolFactory(max_attempts=3)
func = factory.create_tool(ToolSpec(
    name="csv_row_to_json",
    description="Convert CSV row to dict using headers. Takes dict with 'headers' and 'row'.",
    input_type="dict with 'headers' (List[str]) and 'row' (str)",
    output_type="dict mapping each header to corresponding value string",
    test_cases=[
        {"input": {"headers": ["name", "age", "city"], "row": "Alice,30,NYC"},
         "expected": {"name": "Alice", "age": "30", "city": "NYC"}},
        {"input": {"headers": ["id", "score"], "row": "42,98.5"},
         "expected": {"id": "42", "score": "98.5"}},
        {"input": {"headers": ["x"], "row": "hello"}, "expected": {"x": "hello"}},
    ],
))
if func:  # Use the registered tool on new data
    print("\n--- Using registered tool ---")
    print(func({"headers": ["product", "price"], "row": "Widget,9.99"}))


--- Attempt 1/3 for 'csv_row_to_json' ---
Code:
def csv_row_to_json(data):
    headers = data['headers']
    row = data['row']
    
    # Split the row by commas to get individual values
    values = row.split(',')
    
    # Create a dictionary by zipping headers and values
    return dict(zip(headers, values))

Tests:
  Test 1: PASS | {'headers': ['name', 'age', 'city'], 'row': 'Alice,30,NYC'} -> {'name': 'Alice', 'age': '30', 'city': 'NYC'}
  Test 2: PASS | {'headers': ['id', 'score'], 'row': '42,98.5'} -> {'id': '42', 'score': '98.5'}
  Test 3: PASS | {'headers': ['x'], 'row': 'hello'} -> {'x': 'hello'}
REGISTERED 'csv_row_to_json'

--- Using registered tool ---
{'product': 'Widget', 'price': '9.99'}
